# Vision-Based Landslide Forecasting - GPU Training
## RTX 3050 → Google Colab T4 GPU

**Setup steps:**
1. Runtime → Change runtime type → **T4 GPU** select karanna
2. Cell 1 run karanna (Drive mount)
3. Dataset eka Drive ekata upload karala path set karanna
4. Okkoma cells run karanna (Run All)

In [ ]:
# ── Step 1: Mount Google Drive ────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
# GPU check
import tensorflow as tf
print('TF version:', tf.__version__)
print('GPUs:', tf.config.list_physical_devices('GPU'))

In [ ]:
# ── Step 2: Unzip dataset from Drive ─────────────────────────────────────────
# NOTE: Oyage Google Drive eke 'MyDrive/vision_based/' folder eke
#       'dataset_version_2.zip' upload karanna

import zipfile, os

DRIVE_ZIP = '/content/drive/MyDrive/vision_based/dataset_version_2.zip'
EXTRACT_TO = '/content/'

if not os.path.exists('/content/dataset_version_2'):
    print('Extracting dataset...')
    with zipfile.ZipFile(DRIVE_ZIP, 'r') as z:
        z.extractall(EXTRACT_TO)
    print('Done!')
else:
    print('Dataset already extracted.')

# Copy metadata.csv too if needed
import shutil
META_SRC = '/content/drive/MyDrive/vision_based/metadata.csv'
if os.path.exists(META_SRC):
    shutil.copy(META_SRC, '/content/dataset_version_2/metadata.csv')
    print('metadata.csv copied.')

# Verify
files = os.listdir('/content/dataset_version_2')
print(f'Files in dataset_version_2: {files[:5]}...')

In [ ]:
# ── Step 3: Install dependencies ─────────────────────────────────────────────
!pip install -q scikit-learn seaborn matplotlib pandas numpy

In [ ]:
# ── Step 4: Full Training Pipeline ───────────────────────────────────────────
import os
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import matplotlib.pyplot as plt
import seaborn as sns
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (
    Conv2D, MaxPooling2D, Dense, Dropout,
    GlobalAveragePooling2D, BatchNormalization
)

print('GPU:', tf.config.list_physical_devices('GPU'))

# ── Paths ─────────────────────────────────────────────────────────────────────
metadata_path = '/content/dataset_version_2/metadata.csv'

df = pd.read_csv(metadata_path)

# Fix image paths for Colab
# (Local paths 'D:\...' -> '/content/dataset_version_2/...')
def fix_path(p):
    # Get just the filename parts after 'dataset_version_2'
    parts = p.replace('\\', '/').split('dataset_version_2/')
    if len(parts) > 1:
        return '/content/dataset_version_2/' + parts[1]
    return p

df['image_path'] = df['image_path'].apply(fix_path)
df = df[df['image_path'].apply(os.path.exists)].reset_index(drop=True)
print(f'Valid images: {len(df)}')

# ── Geographic split ─────────────────────────────────────────────────────────
gss1 = GroupShuffleSplit(n_splits=1, train_size=0.7, random_state=42)
train_idx, temp_idx = next(gss1.split(df, groups=df['landslide_id']))
df_train = df.iloc[train_idx].copy()
df_temp  = df.iloc[temp_idx].copy()

gss2 = GroupShuffleSplit(n_splits=1, train_size=0.5, random_state=42)
val_idx, test_idx = next(gss2.split(df_temp, groups=df_temp['landslide_id']))
df_val  = df_temp.iloc[val_idx].copy()
df_test = df_temp.iloc[test_idx].copy()

print(f'Train: {len(df_train)} | Val: {len(df_val)} | Test: {len(df_test)}')

# ── Settings (GPU eke 224x224 use karanna puluwan!) ───────────────────────────
IMG_SIZE   = (224, 224)
BATCH_SIZE = 32

df_train['label_str'] = df_train['label'].astype(str)
df_val['label_str']   = df_val['label'].astype(str)
df_test['label_str']  = df_test['label'].astype(str)

# ── Generators ───────────────────────────────────────────────────────────────
cnn_train_gen_obj = ImageDataGenerator(
    rescale=1./255, horizontal_flip=True, vertical_flip=True,
    rotation_range=30, zoom_range=0.2,
    width_shift_range=0.1, height_shift_range=0.1,
    brightness_range=[0.8, 1.2], fill_mode='nearest'
)
cnn_val_gen_obj  = ImageDataGenerator(rescale=1./255)
res_train_gen_obj = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    horizontal_flip=True, vertical_flip=True,
    rotation_range=30, zoom_range=0.2,
    width_shift_range=0.1, height_shift_range=0.1,
    fill_mode='nearest'
)
res_val_gen_obj  = ImageDataGenerator(preprocessing_function=preprocess_input)

def make_gen(tgen, vgen):
    tr = tgen.flow_from_dataframe(df_train, x_col='image_path', y_col='label_str',
        target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='binary', shuffle=True)
    vl = vgen.flow_from_dataframe(df_val, x_col='image_path', y_col='label_str',
        target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='binary', shuffle=False)
    te = vgen.flow_from_dataframe(df_test, x_col='image_path', y_col='label_str',
        target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='binary', shuffle=False)
    return tr, vl, te

cnn_train_gen, cnn_val_gen, cnn_test_gen       = make_gen(cnn_train_gen_obj, cnn_val_gen_obj)
resnet_train_gen, resnet_val_gen, resnet_test_gen = make_gen(res_train_gen_obj, res_val_gen_obj)

# ── Class weights ─────────────────────────────────────────────────────────────
neg, pos = np.bincount(df_train['label'])
total = neg + pos
class_weight = {0: (1/neg)*(total/2.0), 1: (1/pos)*(total/2.0)}
print(f'Class weights -> 0: {class_weight[0]:.3f} | 1: {class_weight[1]:.3f}')

# ── Callbacks ─────────────────────────────────────────────────────────────────
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=5,
        restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5,
        patience=3, min_lr=1e-6, verbose=1)
]

In [ ]:
# ── MODEL 1: Improved Baseline CNN ────────────────────────────────────────────
print('Training Improved Baseline CNN...')

baseline_model = Sequential([
    # Block 1
    Conv2D(32, (3,3), padding='same', activation='relu', input_shape=(224, 224, 3)),
    BatchNormalization(),
    Conv2D(32, (3,3), padding='same', activation='relu'),
    BatchNormalization(),
    MaxPooling2D((2,2)), Dropout(0.25),
    # Block 2
    Conv2D(64, (3,3), padding='same', activation='relu'),
    BatchNormalization(),
    Conv2D(64, (3,3), padding='same', activation='relu'),
    BatchNormalization(),
    MaxPooling2D((2,2)), Dropout(0.25),
    # Block 3
    Conv2D(128, (3,3), padding='same', activation='relu'),
    BatchNormalization(),
    Conv2D(128, (3,3), padding='same', activation='relu'),
    BatchNormalization(),
    MaxPooling2D((2,2)), Dropout(0.25),
    # Block 4
    Conv2D(256, (3,3), padding='same', activation='relu'),
    BatchNormalization(),
    MaxPooling2D((2,2)), Dropout(0.25),
    # Head
    GlobalAveragePooling2D(),
    Dense(256, activation='relu'), BatchNormalization(), Dropout(0.5),
    Dense(64, activation='relu'), Dropout(0.3),
    Dense(1, activation='sigmoid')
], name='Improved_Baseline_CNN')

baseline_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy',
             tf.keras.metrics.Precision(name='precision'),
             tf.keras.metrics.Recall(name='recall'),
             tf.keras.metrics.AUC(name='auc')]
)
baseline_model.summary()

history_baseline = baseline_model.fit(
    cnn_train_gen, validation_data=cnn_val_gen,
    epochs=20, class_weight=class_weight, callbacks=callbacks
)

In [ ]:
# ── MODEL 2: ResNet50 Transfer Learning ───────────────────────────────────────
print('Training ResNet50...')

base_model = tf.keras.applications.ResNet50(
    weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False

x = GlobalAveragePooling2D()(base_model.output)
x = Dense(256, activation='relu')(x)
x = BatchNormalization()(x)
x = Dropout(0.5)(x)
x = Dense(64, activation='relu')(x)
x = Dropout(0.3)(x)
output = Dense(1, activation='sigmoid')(x)

resnet_model = Model(inputs=base_model.input, outputs=output, name='ResNet50_Transfer')
resnet_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy',
             tf.keras.metrics.Precision(name='precision'),
             tf.keras.metrics.Recall(name='recall'),
             tf.keras.metrics.AUC(name='auc')]
)

# Stage 1
print('Stage 1: Training head only...')
history_resnet = resnet_model.fit(
    resnet_train_gen, validation_data=resnet_val_gen,
    epochs=20, class_weight=class_weight, callbacks=callbacks
)

# Stage 2: Fine-tune
print('Stage 2: Fine-tuning last 30 layers...')
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False
resnet_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='binary_crossentropy',
    metrics=['accuracy',
             tf.keras.metrics.Precision(name='precision'),
             tf.keras.metrics.Recall(name='recall'),
             tf.keras.metrics.AUC(name='auc')]
)
resnet_model.fit(
    resnet_train_gen, validation_data=resnet_val_gen,
    epochs=10, class_weight=class_weight, callbacks=callbacks
)

In [ ]:
# ── Evaluation ────────────────────────────────────────────────────────────────
def evaluate_model(model, name, test_gen):
    print(f'\n{"="*40}\n--- Evaluating: {name} ---\n{"="*40}')
    preds  = model.predict(test_gen)
    y_pred = (preds > 0.5).astype(int).flatten()
    y_true = test_gen.classes

    print(classification_report(y_true, y_pred,
          target_names=['No Landslide', 'Landslide']))

    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(6, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['No Landslide', 'Landslide'],
                yticklabels=['No Landslide', 'Landslide'])
    plt.title(f'Confusion Matrix: {name}')
    plt.tight_layout()
    plt.savefig(f'/content/drive/MyDrive/vision_based/confusion_matrix_{name}.png', dpi=150)
    plt.show()

    try:
        auc = roc_auc_score(y_true, preds.flatten())
        print(f'ROC-AUC: {auc:.4f}')
    except Exception as e:
        print(f'AUC: {e}')

evaluate_model(baseline_model, 'Improved_Baseline_CNN', cnn_test_gen)
evaluate_model(resnet_model,   'ResNet50_FineTuned',    resnet_test_gen)

# Save models to Drive
baseline_model.save('/content/drive/MyDrive/vision_based/baseline_cnn_model.h5')
resnet_model.save('/content/drive/MyDrive/vision_based/resnet50_model.h5')
print('Models saved to Google Drive!')